In [ ]:
import os
import struct
import numpy as np

# 2. Define user inputs and problem parameters

This cell defines the main inputs for the SPOD workflow.

It includes:
- grid sizes: `nx`, `ny`, `nz`
- number of time snapshots: `ntime`
- domain lengths: `Lx`, `Ly`
- binary data precision
- block size and overlap for SPOD
- file names for input and output

It also computes some derived quantities such as:
- `ns_one = nx * ny * nz`
- `ns_total = 3 * ns_one`

These values will be used throughout the notebook.

In [ ]:
# =========================================================
# USER INPUTS
# =========================================================
infile = "flattened_data.bin"

nx = 192
ny = 192
nz = 129
ntime = 400

# Domain lengths
Lx = 3.0 * np.pi
Ly = 3.0 * np.pi   # replace if needed

# Binary data type
# Use np.float32 if Fortran REAL was 4 bytes
# Use np.float64 if Fortran wrote 8-byte reals
q_dtype = np.float32

# Working precisions
w_dtype = np.float64
real_work_dtype = np.float64
complex_work_dtype = np.complex128

# Raw binary layout
records_per_timestep = 3
dof = 3

# SPOD block settings
block_size = 50
overlap = 25

# Preprocessing flags
subtract_block_mean = True
use_hann_window = True

# Reconstruction settings
n_modes_keep = 3
row_chunk = 20000

# File names
z_file = "z_coordinates.txt"
q_memmap_file = "Q_matrix.dat"
temp_xf_file = "Xf_temp.dat"

# Derived sizes
ns_one = nx * ny * nz
ns_total = records_per_timestep * ns_one

print("ns_one   =", ns_one)
print("ns_total =", ns_total)

# 3. Define the spatial weight-vector function

This cell defines a function that builds the 1D diagonal weight vector `W_1D`.

The purpose of this vector is to represent the diagonal entries of the spatial weight matrix `W` without explicitly building the full 2D matrix.

The function:
- reads the nonuniform vertical coordinates from file
- computes nodal control-volume weights
- accounts for periodic uniform spacing in `x` and `y`
- expands the weights for all three velocity components `[U, V, W]`

The output of the function is a 1D array of length `3 * nx * ny * nz`.

In [ ]:
def build_W_1D(Lx, Ly, Nx, Ny, Nz, filename='z_coordinates.txt', dof=3, dtype=np.float64):
    if Nz < 2:
        raise ValueError("Nz must be at least 2.")

    try:
        z_coords = np.loadtxt(filename)
    except FileNotFoundError:
        raise FileNotFoundError(f"Could not find '{filename}'.")

    if z_coords.size != Nz:
        raise ValueError(
            f"Grid mismatch: expected Nz={Nz}, file has {z_coords.size} points."
        )

    dx = Lx / Nx
    dy = Ly / Ny

    total_nodes = Nx * Ny * Nz
    W_diag = np.zeros(total_nodes, dtype=dtype)

    idx = 0
    for k in range(Nz):
        if k == 0:
            dz = 0.5 * (z_coords[1] - z_coords[0])
        elif k == Nz - 1:
            dz = 0.5 * (z_coords[k] - z_coords[k - 1])
        else:
            dz = 0.5 * (z_coords[k + 1] - z_coords[k - 1])

        node_volume = dx * dy * dz

        for j in range(Ny):
            for i in range(Nx):
                W_diag[idx] = node_volume
                idx += 1

    domain_volume_calculated = np.sum(W_diag)
    domain_volume_actual = Lx * Ly * (z_coords[-1] - z_coords[0])

    print(f"Total weight sum : {domain_volume_calculated:.12e}")
    print(f"Actual volume    : {domain_volume_actual:.12e}")

    if np.isclose(domain_volume_calculated, domain_volume_actual, rtol=1e-10, atol=1e-12):
        print("SUCCESS: weight matrix matches physical domain volume.")
    else:
        print("WARNING: volumes do not match — check grid inputs.")

    W_1D = np.tile(W_diag, dof)
    return W_1D

# 4. Build and inspect the weight vector

This cell calls the weight-building function and creates `W_1D`.

The result is a 1D array containing one spatial weight for each row of the snapshot matrix.

This cell also prints:
- the total size of `W_1D`
- a few sample values
- a consistency check comparing the sum of the weights to the physical domain volume

In [ ]:
W_1D = build_W_1D(
    Lx=Lx,
    Ly=Ly,
    Nx=nx,
    Ny=ny,
    Nz=nz,
    filename=z_file,
    dof=dof,
    dtype=w_dtype,
)

print("W_1D shape =", W_1D.shape)
print("First 10 weights =", W_1D[:10])

# 5. Define the function for reading one Fortran stream binary block

The original DNS data was written by Fortran using unformatted stream access.

Because the file was written with `ACCESS='STREAM'`, it does not contain Fortran record markers.  
Instead, the binary file is just a continuous stream of raw bytes.

This function reads one raw block at a time by:
- computing how many bytes correspond to one field
- reading exactly that many bytes from the file
- converting the bytes into a NumPy array

This function is the low-level reader used to rebuild the full snapshot matrix from the stream binary file.

In [ ]:
def read_stream_record(f, ns_one, dtype):
    """
    Read one raw block from a Fortran STREAM unformatted file.

    Since ACCESS='STREAM' was used in Fortran, there are no record markers.
    We therefore read exactly ns_one values each time.
    """
    nbytes = ns_one * np.dtype(dtype).itemsize
    data = f.read(nbytes)

    if not data:
        return None

    if len(data) < nbytes:
        raise EOFError(f"Unexpected EOF: expected {nbytes} bytes, got {len(data)}")

    return np.frombuffer(data, dtype=dtype).copy()

# 6. Define the function that builds the full time-domain matrix `Q`

This cell defines the function that converts the raw Fortran binary file into the full 2D snapshot matrix `Q`.

In this matrix:
- rows = spatial DOFs
- columns = time snapshots

Each column is ordered as:
- all `U`
- then all `V`
- then all `W`

The matrix is written to disk as a memmap file.

In [ ]:
def build_Q_memmap(
    infile,
    nx,
    ny,
    nz,
    ntime,
    dtype=np.float32,
    records_per_timestep=3,
    out_memmap="Q_matrix.dat",
):
    ns_one = nx * ny * nz
    ns_total = records_per_timestep * ns_one

    Q = np.memmap(out_memmap, dtype=dtype, mode="w+", shape=(ns_total, ntime))

    with open(infile, "rb") as f:
        for t in range(ntime):
            blocks = []

            for r in range(records_per_timestep):
                rec = read_stream_record(f, ns_one, dtype)

                if rec is None:
                    raise EOFError(
                        f"Unexpected EOF at timestep {t + 1}, record {r + 1}"
                    )

                if rec.size != ns_one:
                    raise ValueError(
                        f"Record size mismatch at timestep {t + 1}, record {r + 1}: "
                        f"got {rec.size}, expected {ns_one}"
                    )

                blocks.append(rec)

            Q[:, t] = np.concatenate(blocks)

    Q.flush()
    return Q

# 7. Build and inspect the full time-domain matrix `Q`

This cell reads the original Fortran binary file and reorganizes it into the full time-domain snapshot matrix `Q`.

The result is stored on disk as a memmap array.

At this stage, the raw binary has been converted into a structured 2D matrix form suitable for SPOD.

In [ ]:
Q = build_Q_memmap(
    infile=infile,
    nx=nx,
    ny=ny,
    nz=nz,
    ntime=ntime,
    dtype=q_dtype,
    records_per_timestep=records_per_timestep,
    out_memmap=q_memmap_file,
)

print("Q stored in:", q_memmap_file)
print("Q shape =", Q.shape)
print("Q dtype =", Q.dtype)

# 8. Re-open the full time-domain matrix from disk

This cell re-opens the memmap file containing `Q`.

This is useful in notebook workflows, because later cells can access `Q` directly from disk without rebuilding it again from the raw binary file.

In [ ]:
Q = np.memmap(q_memmap_file, dtype=q_dtype, mode="r", shape=(ns_total, ntime))
print(Q.shape)

# 9. Define the block-start function

SPOD divides the time series into overlapping blocks.

This cell defines the function that computes the valid starting indices of all complete blocks based on:
- total number of snapshots
- block size
- overlap

In [ ]:
def get_block_starts(ntime, block_size, overlap):
    if block_size <= 1:
        raise ValueError("block_size must be at least 2.")
    if overlap < 0:
        raise ValueError("overlap must be non-negative.")
    if overlap >= block_size:
        raise ValueError("overlap must be smaller than block_size.")
    if block_size > ntime:
        raise ValueError("block_size cannot exceed ntime.")

    step = block_size - overlap
    starts = list(range(0, ntime - block_size + 1, step))
    return starts

# 10. Compute and inspect the SPOD block layout

This cell computes:
- the list of block starting indices
- the number of blocks
- the number of frequencies returned by the real FFT

These values determine the structure of the later SPOD matrices.

In [ ]:
starts = get_block_starts(ntime, block_size, overlap)
nblk = len(starts)
nfreq = block_size // 2 + 1

print("starts =", starts)
print("nblk   =", nblk)
print("nfreq  =", nfreq)

# 11. Extract one time block from `Q`

This cell selects one block from the full time-domain matrix.

The block has shape:
- rows = spatial DOFs
- columns = time samples within the block

At this point, the data is still in the physical time domain.

In [ ]:
m = 0
start = starts[m]
stop = start + block_size

Q_block = np.asarray(Q[:, start:stop], dtype=real_work_dtype)

print("Block index :", m)
print("start, stop :", start, stop)
print("Q_block shape =", Q_block.shape)

# 12. Subtract the block mean

This cell optionally subtracts the time mean from each row of the selected block.

This converts the block into fluctuation data, which is usually preferred before FFT and SPOD.

In [ ]:
Q_block_centered = Q_block.copy()

if subtract_block_mean:
    Q_block_centered = Q_block_centered - Q_block_centered.mean(axis=1, keepdims=True)

print("Centered block shape =", Q_block_centered.shape)

# 13. Create the temporal window

This cell creates the time-window function used before FFT.

If `use_hann_window` is true, a Hann window is used.
Otherwise, an all-ones window is used.

The purpose is to reduce spectral leakage.

In [ ]:
if use_hann_window:
    window = np.hanning(block_size).astype(real_work_dtype)
else:
    window = np.ones(block_size, dtype=real_work_dtype)

print("window shape =", window.shape)
print(window[:10])

# 14. Apply the window to one block

This cell multiplies every row of the block by the same temporal window.

This tapers the signal at the beginning and end of the block before FFT.

In [ ]:
Q_block_windowed = Q_block_centered * window[np.newaxis, :]
print("Q_block_windowed shape =", Q_block_windowed.shape)

# 15. Compute the FFT of one block

This cell applies the FFT along the time direction of the selected block.

The output is now in the frequency domain.

Its shape is:
- rows = spatial DOFs
- columns = retained frequencies

In [ ]:
Qhat_block = np.fft.rfft(Q_block_windowed, axis=1).astype(complex_work_dtype)

print("Qhat_block shape =", Qhat_block.shape)
print("Expected nfreq   =", nfreq)

# 16. Inspect one frequency vector from one block

This cell extracts one frequency column from the FFT of one block.

The result is a complex vector of length equal to the number of spatial DOFs.

This represents the flow field at one selected frequency for one selected block.

In [ ]:
j = 0
freq_vector = Qhat_block[:, j]

print("freq index =", j)
print("freq_vector shape =", freq_vector.shape)
print("first 5 entries =", freq_vector[:5])

# 17. Define the function that builds one frequency-wise matrix `X(f)`

For one chosen frequency index, this function:
- loops over all blocks
- extracts the block from `Q`
- optionally subtracts the block mean
- applies the temporal window
- computes the FFT
- stores the selected frequency into the corresponding block column

The result is the SPOD matrix `X(f)` with shape:
- rows = spatial DOFs
- columns = blocks

In [ ]:
def build_single_frequency_matrix(
    q_memmap_file,
    shape_q,
    starts,
    block_size,
    freq_index,
    q_dtype=np.float32,
    real_dtype=np.float64,
    complex_dtype=np.complex128,
    subtract_block_mean=True,
    use_hann_window=True,
    temp_xf_file="Xf_temp.dat",
):
    ns, ntime = shape_q
    nblk = len(starts)

    Q = np.memmap(q_memmap_file, dtype=q_dtype, mode="r", shape=(ns, ntime))
    Xf = np.memmap(temp_xf_file, dtype=complex_dtype, mode="w+", shape=(ns, nblk))

    if use_hann_window:
        window = np.hanning(block_size).astype(real_dtype)
    else:
        window = np.ones(block_size, dtype=real_dtype)

    for m, start in enumerate(starts):
        stop = start + block_size
        Q_block = np.asarray(Q[:, start:stop], dtype=real_dtype)

        if subtract_block_mean:
            Q_block = Q_block - Q_block.mean(axis=1, keepdims=True)

        Q_block = Q_block * window[np.newaxis, :]
        Qhat_block = np.fft.rfft(Q_block, axis=1).astype(complex_dtype)

        Xf[:, m] = Qhat_block[:, freq_index]

    Xf.flush()
    return Xf

# 18. Build and inspect one frequency-wise matrix `X(f)`

This cell constructs `X(f)` for one selected frequency.

This matrix contains:
- rows = spatial DOFs
- columns = blocks

This is the matrix that will be used to build the reduced SPOD matrix at that frequency.

In [ ]:
j = 0

Xf = build_single_frequency_matrix(
    q_memmap_file=q_memmap_file,
    shape_q=(ns_total, ntime),
    starts=starts,
    block_size=block_size,
    freq_index=j,
    q_dtype=q_dtype,
    real_dtype=real_work_dtype,
    complex_dtype=complex_work_dtype,
    subtract_block_mean=subtract_block_mean,
    use_hann_window=use_hann_window,
    temp_xf_file=temp_xf_file,
)

print("Xf shape =", Xf.shape)
print("Xf dtype =", Xf.dtype)

# 19. Define the reduced SPOD matrix function

This function computes the reduced matrix

\[
R(f) = X(f)^* W X(f)
\]

without explicitly building the full diagonal matrix `W`.

Instead, it uses the 1D weight vector `W_1D` and weighted row scaling.

In [ ]:
def compute_reduced_matrix_from_Xf(
    xf_memmap_file,
    shape_xf,
    W_1D,
    complex_dtype=np.complex128,
    out_dtype=np.complex128,
    row_chunk=20000,
):
    ns, nblk = shape_xf

    if W_1D.shape[0] != ns:
        raise ValueError(
            f"W_1D length mismatch: got {W_1D.shape[0]}, expected {ns}"
        )

    Xf = np.memmap(xf_memmap_file, dtype=complex_dtype, mode="r", shape=(ns, nblk))
    Rf = np.zeros((nblk, nblk), dtype=out_dtype)

    for i0 in range(0, ns, row_chunk):
        i1 = min(i0 + row_chunk, ns)

        X_chunk = np.asarray(Xf[i0:i1, :], dtype=out_dtype)
        W_chunk = np.asarray(W_1D[i0:i1], dtype=np.float64)

        WX_chunk = W_chunk[:, np.newaxis] * X_chunk
        Rf += X_chunk.conj().T @ WX_chunk

    return Rf

# 20. Compute and inspect one reduced matrix `R(f)`

This cell computes the reduced SPOD matrix for one frequency.

The output is a small matrix of size:
- number of blocks × number of blocks

This is the matrix on which the POD eigenvalue decomposition is performed.

In [ ]:
Rf = compute_reduced_matrix_from_Xf(
    xf_memmap_file=temp_xf_file,
    shape_xf=(ns_total, nblk),
    W_1D=W_1D,
    complex_dtype=complex_work_dtype,
    out_dtype=complex_work_dtype,
    row_chunk=row_chunk,
)

print("Rf shape =", Rf.shape)
print("Rf[0:3,0:3] =\n", Rf[:3, :3])

# 21. Define the reduced eigenproblem solver

This function solves the Hermitian eigenvalue problem for the reduced matrix.

It:
- computes eigenvalues and eigenvectors
- sorts the eigenvalues in descending order
- reorders the eigenvectors
- clips tiny negative roundoff values to zero

In [ ]:
def solve_reduced_eigenproblem(Rf, tol=1e-12):
    eigvals, eigvecs = np.linalg.eigh(Rf)

    idx = np.argsort(eigvals.real)[::-1]
    eigvals = eigvals[idx].real
    eigvecs = eigvecs[:, idx]

    eigvals[np.abs(eigvals) < tol] = 0.0
    eigvals[eigvals < 0.0] = 0.0

    return eigvals, eigvecs

# 22. Solve the reduced eigenproblem for one frequency

This cell computes the eigenvalues and reduced eigenvectors of the reduced SPOD matrix.

The eigenvalues are the modal energies.
The eigenvectors are the reduced block-space coefficients used to reconstruct the actual spatial modes.

In [ ]:
eigvals, eigvecs = solve_reduced_eigenproblem(Rf)

print("eigvals shape =", eigvals.shape)
print("eigvecs shape =", eigvecs.shape)
print("first 10 eigvals =", eigvals[:10])

# 23. Define the spatial mode reconstruction function

This function reconstructs the actual spatial SPOD modes using:

$$
\Phi(f) = X(f)\Psi(f)\,(N_{blk}\Lambda)^{-1/2}
$$

It takes:
- the frequency-wise matrix `X(f)`
- the reduced eigenvectors
- the eigenvalues

and returns the spatial modes stored on disk.

In [ ]:
def reconstruct_spod_modes(
    xf_memmap_file,
    shape_xf,
    eigvals,
    eigvecs,
    nblk,
    n_modes_keep=None,
    complex_dtype=np.complex128,
    out_dtype=np.complex128,
    row_chunk=20000,
    mode_file="Phi_freq_0000.dat",
    tol=1e-12,
):
    ns, nblk_check = shape_xf
    if nblk_check != nblk:
        raise ValueError("shape_xf second dimension does not match nblk.")

    if n_modes_keep is None:
        n_modes_keep = nblk

    n_modes_keep = min(n_modes_keep, nblk)

    valid = eigvals[:n_modes_keep] > tol
    kept_indices = np.where(valid)[0]
    n_keep_valid = kept_indices.size

    Xf = np.memmap(xf_memmap_file, dtype=complex_dtype, mode="r", shape=(ns, nblk))

    Phi = np.memmap(
        mode_file,
        dtype=out_dtype,
        mode="w+",
        shape=(ns, n_keep_valid),
    )
    Phi[:] = 0.0 + 0.0j

    if n_keep_valid == 0:
        Phi.flush()
        return Phi, kept_indices

    lam_keep = eigvals[kept_indices]
    psi_keep = eigvecs[:, kept_indices]

    coeff = psi_keep / np.sqrt(nblk * lam_keep)[np.newaxis, :]

    for i0 in range(0, ns, row_chunk):
        i1 = min(i0 + row_chunk, ns)

        X_chunk = np.asarray(Xf[i0:i1, :], dtype=out_dtype)
        Phi[i0:i1, :] = X_chunk @ coeff

    Phi.flush()
    return Phi, kept_indices

# 24. Reconstruct and inspect the spatial modes for one frequency

This cell reconstructs the actual SPOD modes for the selected frequency.

Each column of `Phi` is one spatial SPOD mode.
Only the first few requested modes are kept.

In [ ]:
Phi, kept_idx = reconstruct_spod_modes(
    xf_memmap_file=temp_xf_file,
    shape_xf=(ns_total, nblk),
    eigvals=eigvals,
    eigvecs=eigvecs,
    nblk=nblk,
    n_modes_keep=n_modes_keep,
    complex_dtype=complex_work_dtype,
    out_dtype=complex_work_dtype,
    row_chunk=row_chunk,
    mode_file="Phi_freq_0000.dat",
)

print("Phi shape =", Phi.shape)
print("kept mode indices =", kept_idx)

# 25. Define a wrapper that runs the full SPOD workflow for one frequency

This function combines the main SPOD steps for a single frequency:

1. build `X(f)`
2. compute `R(f)`
3. solve the reduced eigenproblem
4. reconstruct the spatial modes

This is convenient for debugging or studying one frequency at a time.

In [ ]:
def run_one_frequency_spod(
    freq_index,
    q_memmap_file,
    shape_q,
    W_1D,
    starts,
    block_size,
    q_dtype=np.float32,
    real_dtype=np.float64,
    complex_dtype=np.complex128,
    subtract_block_mean=True,
    use_hann_window=True,
    row_chunk=20000,
    n_modes_keep=3,
    temp_xf_file="Xf_temp.dat",
    mode_file=None,
):
    ns, ntime = shape_q
    nblk = len(starts)

    if mode_file is None:
        mode_file = f"Phi_freq_{freq_index:04d}.dat"

    Xf = build_single_frequency_matrix(
        q_memmap_file=q_memmap_file,
        shape_q=shape_q,
        starts=starts,
        block_size=block_size,
        freq_index=freq_index,
        q_dtype=q_dtype,
        real_dtype=real_dtype,
        complex_dtype=complex_dtype,
        subtract_block_mean=subtract_block_mean,
        use_hann_window=use_hann_window,
        temp_xf_file=temp_xf_file,
    )

    Rf = compute_reduced_matrix_from_Xf(
        xf_memmap_file=temp_xf_file,
        shape_xf=(ns, nblk),
        W_1D=W_1D,
        complex_dtype=complex_dtype,
        out_dtype=complex_dtype,
        row_chunk=row_chunk,
    )

    eigvals, eigvecs = solve_reduced_eigenproblem(Rf)

    Phi, kept_idx = reconstruct_spod_modes(
        xf_memmap_file=temp_xf_file,
        shape_xf=(ns, nblk),
        eigvals=eigvals,
        eigvecs=eigvecs,
        nblk=nblk,
        n_modes_keep=n_modes_keep,
        complex_dtype=complex_dtype,
        out_dtype=complex_dtype,
        row_chunk=row_chunk,
        mode_file=mode_file,
    )

    return Xf, Rf, eigvals, eigvecs, Phi, kept_idx

# 26. Run the full SPOD workflow for one selected frequency

This cell executes the complete SPOD pipeline for one frequency index.

It returns:
- the frequency-wise matrix `X(f)`
- the reduced matrix `R(f)`
- eigenvalues
- reduced eigenvectors
- reconstructed spatial modes

This is the best cell to test before looping over all frequencies.

In [ ]:
freq_index = 0

Xf, Rf, eigvals, eigvecs, Phi, kept_idx = run_one_frequency_spod(
    freq_index=freq_index,
    q_memmap_file=q_memmap_file,
    shape_q=(ns_total, ntime),
    W_1D=W_1D,
    starts=starts,
    block_size=block_size,
    q_dtype=q_dtype,
    real_dtype=real_work_dtype,
    complex_dtype=complex_work_dtype,
    subtract_block_mean=subtract_block_mean,
    use_hann_window=use_hann_window,
    row_chunk=row_chunk,
    n_modes_keep=n_modes_keep,
    temp_xf_file=temp_xf_file,
    mode_file=f"Phi_freq_{freq_index:04d}.dat",
)

print("Finished frequency", freq_index)
print("Rf shape =", Rf.shape)
print("Phi shape =", Phi.shape)
print("First few eigvals =", eigvals[:10])

# 27. Optional loop over all frequencies

This cell shows how to repeat the full one-frequency SPOD workflow for every retained frequency.

This should only be done after the single-frequency pipeline has been tested and verified.

In [ ]:
all_results = []

for freq_index in range(nfreq):
    print("\nRunning frequency", freq_index)

    Xf, Rf, eigvals, eigvecs, Phi, kept_idx = run_one_frequency_spod(
        freq_index=freq_index,
        q_memmap_file=q_memmap_file,
        shape_q=(ns_total, ntime),
        W_1D=W_1D,
        starts=starts,
        block_size=block_size,
        q_dtype=q_dtype,
        real_dtype=real_work_dtype,
        complex_dtype=complex_work_dtype,
        subtract_block_mean=subtract_block_mean,
        use_hann_window=use_hann_window,
        row_chunk=row_chunk,
        n_modes_keep=n_modes_keep,
        temp_xf_file=temp_xf_file,
        mode_file=f"Phi_freq_{freq_index:04d}.dat",
    )

    all_results.append((Rf, eigvals, kept_idx))